<a href="https://colab.research.google.com/github/gracella12/PP_Assignment/blob/main/PP_Assignment01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Buatlah fungsi untuk melakukan preprocessing dan ekstraksi ciri TF-ID pada data teks tanpa menggunakan library. Kerjakan menggunakan bahasa python melalui google colab

In [1]:
import pandas as pd
import re

In [2]:
data = pd.read_csv("dataset_detik.csv")
data

,Isi Artikel
0,"Menteri Koordinator Kemaritiman dan Investasi,..."
1,Menteri Keuangan Sri Mulyani Indrawati merasa ...
2,"Terakhir, ia mengimbau masyarakat supaya menga..."


# Preprocessing

In [3]:
import re

def simple_stem(word):
    prefixes = ['meng', 'meny', 'men', 'mem', 'me',
                'peng', 'peny', 'pen', 'pem',
                'ber', 'ter', 'per', 'di', 'ke', 'se']

    suffixes = ['lah', 'kah', 'nya', 'pun', 'ku', 'mu',
                'kan', 'an', 'i']

    original_word = word

    # Remove prefix (only if word length > 4)
    for prefix in prefixes:
        if word.startswith(prefix) and len(word) > len(prefix) + 2:
            word = word[len(prefix):]
            break

    # Remove suffix (only if word length > 3)
    for suffix in suffixes:
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            word = word[:-len(suffix)]
            break

    # Prevent empty result
    if len(word) <= 2:
        return original_word

    return word


def preprocessing(text):

    #detele "klik halaman berikutnya untuk membaca artikel selengkapnya"
    text = re.sub(r'klik halaman berikutnya untuk membaca artikel selengkapnya', '', text, flags=re.IGNORECASE)

    # Case folding
    text = text.lower()

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Stopword removal
    stopwords = {
        "yang", "untuk", "dengan", "pada", "dan", "di", "dari",
        "ke", "sebagai", "adalah", "ini", "itu", "saya", "kamu",
        "dia", "kami", "mereka", "akan", "telah", "dalam",
        "juga", "seperti", "karena", "oleh", "saat", "sudah",
        "belum", "masih", "harus"
    }

    words = text.split()
    words = [word for word in words if word not in stopwords]

    # Stemming
    words = [simple_stem(word) for word in words]

    return " ".join(words)

In [4]:
#Hasil preprocessing
data['clean_text'] = data['Isi Artikel'].astype(str).apply(preprocessing)
print(data['clean_text'].head())

0    ter koordinator maritim investas luhut binsar ...
1    ter uang sri mulyan indrawat rasa hilang legen...
2    akhir ia imbau masyarakat supaya acu informas ...
Name: clean_text, dtype: object


# TF-IDF

In [5]:
def term_count(document):
    words = document.split()
    freq = {}

    for word in words:
        if word in freq:
            freq[word] += 1
        else:
            freq[word] = 1

    return freq


def term_frequency(document):

    words = document.split()
    total_words = len(words)
    tf_dict = {}

    for word in words:
        if word in tf_dict:
            tf_dict[word] += 1
        else:
            tf_dict[word] = 1

    # Normalisasi (bagi total kata)
    for word in tf_dict:
        tf_dict[word] = tf_dict[word] / total_words

    return tf_dict

In [6]:
for index, row in data.iterrows():
    clean_text = preprocessing(str(row['Isi Artikel']))

    raw_tf = term_count(clean_text)        # sebelum normalisasi
    normalized_tf = term_frequency(clean_text)  # setelah normalisasi

    print(f"Row {index}")
    print("Clean Text:", clean_text)
    print("TF (Raw Count):", raw_tf)
    print("TF (Normalized):", normalized_tf)
    print("-" * 50)

Row 0
Clean Text: ter koordinator maritim investas luhut binsar pandjait buka suara soal masa transis blok rok luhut inta masa transis bisa laku lebih cepatluhut ata belum inta masa transis bisa waktu tahun namun kal inta masa transis percepatya suruh dua tahun tap minta percepat kata luhut kantor jumat luhut inta agar tamina tak usah repotrepot gant formula kontrak rilis chevronsudah kita lesai tad biar paka saja formula punya chevron ngapain gantigant bilang tad supaya bicarain biar nggak lama lag car car cara lain kata luhut bagaimana kondis blok rok kin klik halam ikut
TF (Raw Count): {'ter': 1, 'koordinator': 1, 'maritim': 1, 'investas': 1, 'luhut': 5, 'binsar': 1, 'pandjait': 1, 'buka': 1, 'suara': 1, 'soal': 1, 'masa': 4, 'transis': 4, 'blok': 2, 'rok': 2, 'inta': 4, 'bisa': 2, 'laku': 1, 'lebih': 1, 'cepatluhut': 1, 'ata': 1, 'belum': 1, 'waktu': 1, 'tahun': 2, 'namun': 1, 'kal': 1, 'percepatya': 1, 'suruh': 1, 'dua': 1, 'tap': 1, 'minta': 1, 'percepat': 1, 'kata': 2, 'kantor':

In [7]:
def inverse_document_frequency(corpus):
    import math

    N = len(corpus)
    idf_dict = {}

    # Hitung document frequency
    for document in corpus:
        words = set(document.split())
        for word in words:
            if word in idf_dict:
                idf_dict[word] += 1
            else:
                idf_dict[word] = 1

    # Hitung IDF (smooth version)
    for word in idf_dict:
        df = idf_dict[word]
        idf_dict[word] = math.log((N + 1) / (df + 1)) + 1

    return idf_dict

In [8]:
corpus = data['clean_text'].tolist()
idf = inverse_document_frequency(corpus)

In [9]:
sorted_idf = sorted(idf.items(), key=lambda x: x[1], reverse=True)

print("Top 20 Highest IDF Words:\n")
for word, value in sorted_idf[:20]:
    print(f"{word}: {value:.4f}")

Top 20 Highest IDF Words:

buka: 1.6931
usah: 1.6931
lesai: 1.6931
car: 1.6931
tad: 1.6931
klik: 1.6931
punya: 1.6931
tap: 1.6931
percepat: 1.6931
chevron: 1.6931
gantigant: 1.6931
lain: 1.6931
kin: 1.6931
agar: 1.6931
koordinator: 1.6931
rok: 1.6931
chevronsudah: 1.6931
biar: 1.6931
bilang: 1.6931
soal: 1.6931


In [10]:
#TF-IDF
def tf_idf(document, idf_dict):
    tf_dict = term_frequency(document)
    tf_idf_dict = {}

    for word, tf_value in tf_dict.items():
        idf_value = idf_dict.get(word, 0)
        tf_idf_dict[word] = tf_value * idf_value

    return tf_idf_dict


In [11]:
data['tf_idf'] = data['clean_text'].apply(lambda x: tf_idf(x, idf))

In [12]:
for index, row in data.iterrows():
    sorted_tfidf = sorted(row['tf_idf'].items(), key=lambda x: x[1], reverse=True)

    print(f"\nRow {index} - Top 10 TF-IDF Words:")
    for word, score in sorted_tfidf[:10]:
        print(f"{word}: {score:.4f}")


Row 0 - Top 10 TF-IDF Words:
luhut: 0.0930
masa: 0.0744
transis: 0.0744
inta: 0.0566
blok: 0.0372
rok: 0.0372
tahun: 0.0372
formula: 0.0372
tad: 0.0372
biar: 0.0372

Row 1 - Top 10 TF-IDF Words:
kita: 0.0784
mulyan: 0.0344
kobe: 0.0344
bryant: 0.0344
baik: 0.0344
bag: 0.0344
sri: 0.0258
rasa: 0.0258
sikap: 0.0258
tidak: 0.0258

Row 2 - Top 10 TF-IDF Words:
corona: 0.1426
virus: 0.1381
hoax: 0.0936
sinformas: 0.0757
pasien: 0.0446
china: 0.0312
orang: 0.0203
inggal: 0.0203
dr: 0.0178
duga: 0.0178
